In [ ]:
SEED        = 17
BATCH_SIZE  = 128
EPOCHS      = 50
EPS         = 8/255
ALPHA       = 2/255
PGD_STEPS   = 20
NUM_CLASSES = 10
DEVICE      = "cuda"

MODEL_NAMES = ["resnet34", "vgg11", "resnet18", "vgg16", "densenet"]
MODEL_PATHS = {name: f"../models/{name}.pth" for name in MODEL_NAMES}
ADV_PATHS   = {name: f"../adv_examples/{name}.pt" for name in MODEL_NAMES}

In [3]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from tqdm.auto import tqdm

torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)

/home/mshubham/folder/aiml-project/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import torchvision.models as tvm

# using torchvision pretrained architectures, just changing final layer for 10 classes

def get_resnet34():
    m = tvm.resnet34(weights=None)
    m.fc = nn.Linear(512, 10)
    return m

def get_vgg11():
    m = tvm.vgg11(weights=None)
    m.classifier[6] = nn.Linear(4096, 10)
    return m

def get_resnet18():
    m = tvm.resnet18(weights=None)
    m.fc = nn.Linear(512, 10)
    return m

def get_vgg16():
    m = tvm.vgg16(weights=None)
    m.classifier[6] = nn.Linear(4096, 10)
    return m

def get_densenet():
    m = tvm.densenet121(weights=None)
    m.classifier = nn.Linear(1024, 10)
    return m

def get_model(name):
    return {
        "resnet34": get_resnet34,
        "vgg11": get_vgg11,
        "resnet18": get_resnet18,
        "vgg16": get_vgg16,
        "densenet": get_densenet
    }[name]()

In [5]:
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
])

trainloader = torch.utils.data.DataLoader(
    torchvision.datasets.CIFAR10('../data', train=True,  download=True, transform=transform_train),
    batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)

testloader = torch.utils.data.DataLoader(
    torchvision.datasets.CIFAR10('../data', train=False, download=True, transform=transform_test),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

In [6]:
def train_model(name):
    print(f"\n--- Training {name} ---")
    
    model = get_model(name).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    if name in ["vgg11", "vgg16"]:
        lr = 1e-4
    else:
        lr = 1e-3
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0
        for images, labels in tqdm(trainloader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()
        print(f"  Loss: {total_loss/len(trainloader):.4f}")

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in testloader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            correct += (model(images).argmax(1) == labels).sum().item()
            total   += labels.size(0)
    acc = 100 * correct / total
    print(f"  Test accuracy: {acc:.2f}%")

    torch.save(model.state_dict(), MODEL_PATHS[name])
    print(f"  Saved → {MODEL_PATHS[name]}")

In [7]:
os.makedirs("../models", exist_ok=True)
train_model("densenet")


--- Training densenet ---


Epoch 1/50: 100%|██████████| 391/391 [00:25<00:00, 15.15it/s]


  Loss: 1.5265


Epoch 2/50: 100%|██████████| 391/391 [00:24<00:00, 16.05it/s]


  Loss: 1.1689


...

Epoch 49/50: 100%|██████████| 391/391 [00:24<00:00, 15.76it/s]


  Loss: 0.0540


Epoch 50/50: 100%|██████████| 391/391 [00:24<00:00, 15.77it/s]

  Loss: 0.0520


  Test accuracy: 86.32%
  Saved → ../models/densenet.pth


In [8]:
train_model("vgg16")


--- Training vgg16 ---


Epoch 1/50: 100%|██████████| 391/391 [01:02<00:00,  6.24it/s]


  Loss: 1.9549


Epoch 2/50: 100%|██████████| 391/391 [01:02<00:00,  6.25it/s]


  Loss: 1.5252


...

Epoch 49/50: 100%|██████████| 391/391 [01:02<00:00,  6.25it/s]


  Loss: 0.0733


Epoch 50/50: 100%|██████████| 391/391 [01:02<00:00,  6.25it/s]

  Loss: 0.0738


  Test accuracy: 85.08%
  Saved → ../models/vgg16.pth


In [9]:
train_model("vgg11")


--- Training vgg11 ---


Epoch 1/50: 100%|██████████| 391/391 [00:51<00:00,  7.63it/s]


  Loss: 1.8071


Epoch 2/50: 100%|██████████| 391/391 [00:51<00:00,  7.63it/s]


  Loss: 1.4035


...

Epoch 49/50: 100%|██████████| 391/391 [00:51<00:00,  7.58it/s]


  Loss: 0.0984


Epoch 50/50: 100%|██████████| 391/391 [00:51<00:00,  7.59it/s]

  Loss: 0.0999


  Test accuracy: 82.48%
  Saved → ../models/vgg11.pth


In [10]:
train_model("resnet18")


--- Training resnet18 ---


Epoch 1/50: 100%|██████████| 391/391 [00:07<00:00, 49.97it/s]


  Loss: 1.5507


Epoch 2/50: 100%|██████████| 391/391 [00:07<00:00, 50.61it/s]


  Loss: 1.1844


...

  Loss: 0.1388


Epoch 49/50: 100%|██████████| 391/391 [00:07<00:00, 49.69it/s]


  Loss: 0.1363


Epoch 50/50: 100%|██████████| 391/391 [00:07<00:00, 49.29it/s]


  Loss: 0.1330
  Test accuracy: 85.58%
  Saved → ../models/resnet18.pth


In [ ]:
train_model("resnet34")


--- Training resnet34 ---


Epoch 1/50: 100%|██████████| 391/391 [00:13<00:00, 28.36it/s]


  Loss: 1.6280


Epoch 2/50: 100%|██████████| 391/391 [00:13<00:00, 28.58it/s]


  Loss: 1.2392


...

  Loss: 0.1394


Epoch 50/50: 100%|██████████| 391/391 [00:13<00:00, 29.35it/s]

  Loss: 0.1366


  Test accuracy: 85.61%
  Saved → ../models/resnet34.pth


: 